In [1]:
import pandas as pd

fact_flights   = pd.read_csv("../data/processed/model/fact_flights.csv", parse_dates=["departure_time", "arrival_time"])
fact_bookings  = pd.read_csv("../data/processed/model/fact_bookings.csv")
dim_date       = pd.read_csv("../data/processed/model/dim_date.csv", parse_dates=["date"])
dim_airline    = pd.read_csv("../data/processed/model/dim_airline.csv")
dim_route      = pd.read_csv("../data/processed/model/dim_route.csv")

In [ ]:
avg_duration_overall = fact_flights.loc[~fact_flights["is_corrupted_time"], "duration_minutes"].mean()
print(f"Average flight duration (overall): {avg_duration_overall:.1f} minutes")


avg_duration_by_route = (
    fact_flights[~fact_flights["is_corrupted_time"]]
    .merge(dim_route, on="route_id")
    .groupby("route_label")["duration_minutes"]
    .mean()
    .sort_values(ascending=False)
    .round(1)
)
print(avg_duration_by_route)

Average flight duration (overall): 164.5 minutes
route_label
BLR → MAA    187.3
HYD → DEL    185.4
BOM → MAA    183.4
BOM → HYD    180.8
DEL → MAA    178.8
DEL → BOM    178.1
MAA → BOM    176.8
BOM → BLR    176.5
DEL → HYD    174.8
MAA → BLR    172.8
CCU → MAA    171.3
DEL → BLR    170.0
BOM → CCU    169.5
CCU → BLR    167.4
HYD → CCU    167.2
CCU → BOM    164.0
HYD → BLR    162.8
MAA → HYD    160.0
MAA → CCU    158.5
BLR → DEL    156.9
MAA → DEL    156.0
BLR → HYD    155.7
BOM → DEL    153.8
CCU → DEL    153.6
HYD → MAA    152.8
CCU → HYD    152.2
DEL → CCU    151.7
HYD → BOM    149.9
BLR → BOM    147.7
BLR → CCU    128.5
Name: duration_minutes, dtype: float64


In [3]:
route_traffic = (
    fact_flights.merge(dim_route, on="route_id")
    .groupby("route_label")
    .size()
    .sort_values(ascending=False)
    .rename("flight_count")
)
print(route_traffic)

route_label
BOM → CCU    90
CCU → DEL    72
MAA → BLR    65
BLR → BOM    60
HYD → MAA    57
DEL → HYD    54
HYD → DEL    42
BOM → DEL    39
CCU → BOM    33
DEL → BLR    29
DEL → BOM    28
HYD → BOM    27
BOM → MAA    27
BOM → HYD    26
MAA → DEL    26
HYD → CCU    26
DEL → CCU    26
HYD → BLR    26
MAA → CCU    24
CCU → MAA    24
BOM → BLR    23
DEL → MAA    23
BLR → CCU    21
CCU → HYD    21
MAA → BOM    21
MAA → HYD    21
CCU → BLR    20
BLR → DEL    19
BLR → HYD    19
BLR → MAA    16
Name: flight_count, dtype: int64


In [ ]:
anomalies = pd.DataFrame({
    "anomaly_type": ["Corrupted timestamp (arrival before departure)", "Overnight flight", "Unknown airline"],
    "count": [
        fact_flights["is_corrupted_time"].sum(),
        fact_flights["is_overnight"].sum(),
        (fact_flights.merge(dim_airline, on="airline_id")["airline"] == "UNKNOWN").sum()
    ]
})
print(anomalies)


route_stats = fact_flights.merge(dim_route, on="route_id").groupby("route_label")["duration_minutes"].agg(["mean", "std"])
merged = fact_flights.merge(dim_route, on="route_id").merge(route_stats, on="route_label")
merged["duration_zscore"] = (merged["duration_minutes"] - merged["mean"]) / merged["std"]
duration_outliers = merged[merged["duration_zscore"].abs() > 2]  # >2 std devs from route average
print(f"Duration outliers (>2 std dev from route mean): {len(duration_outliers)}")

                                     anomaly_type  count
0  Corrupted timestamp (arrival before departure)      1
1                                Overnight flight    122
2                                 Unknown airline     69
Duration outliers (>2 std dev from route mean): 1


In [5]:
airline_distribution = (
    fact_flights.merge(dim_airline, on="airline_id")
    .groupby("airline")
    .size()
    .sort_values(ascending=False)
    .rename("flight_count")
)
airline_distribution_pct = (airline_distribution / airline_distribution.sum() * 100).round(1)
print(pd.concat([airline_distribution, airline_distribution_pct.rename("pct")], axis=1))

           flight_count   pct
airline                      
Indigo              249  24.8
Spicejet            236  23.5
Air India           233  23.2
Vistara             218  21.7
UNKNOWN              69   6.9


In [ ]:

fb_with_route = fact_bookings.merge(fact_flights[["flight_id", "route_id"]], on="flight_id").merge(dim_route, on="route_id")
revenue_by_route = fb_with_route.groupby("route_label")["amount"].sum().sort_values(ascending=False).round(0)


booking_status_dist = fact_bookings["status"].value_counts()


payment_method_dist = fact_bookings["payment_method"].value_counts()


fact_flights_with_date = fact_flights.merge(dim_date, left_on="dep_date_id", right_on="date_id")
traffic_by_dow = fact_flights_with_date.groupby("day_name").size().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
)

In [7]:
import os
os.makedirs("../data/processed/kpi", exist_ok=True)

avg_duration_by_route.to_csv("../data/processed/kpi/avg_duration_by_route.csv")
route_traffic.to_csv("../data/processed/kpi/route_traffic.csv")
anomalies.to_csv("../data/processed/kpi/anomalies.csv", index=False)
duration_outliers.to_csv("../data/processed/kpi/duration_outliers.csv", index=False)
airline_distribution.to_csv("../data/processed/kpi/airline_distribution.csv")
revenue_by_route.to_csv("../data/processed/kpi/revenue_by_route.csv")
booking_status_dist.to_csv("../data/processed/kpi/booking_status_dist.csv")
traffic_by_dow.to_csv("../data/processed/kpi/traffic_by_dow.csv")